In [1]:
import pandas as pd

In [2]:
data_pib = pd.read_csv("../data/5_BANCO MUNDIAL_PIB PER CAPITA.csv", skiprows=4)
print(data_pib.shape)

(266, 71)


In [8]:
data_pib.info()

<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   pais    266 non-null    str    
 1   iso3    266 non-null    str    
 2   2015    260 non-null    float64
 3   2016    259 non-null    float64
 4   2017    259 non-null    float64
 5   2018    259 non-null    float64
 6   2019    259 non-null    float64
 7   2020    258 non-null    float64
 8   2021    258 non-null    float64
 9   2022    257 non-null    float64
 10  2023    252 non-null    float64
 11  2024    248 non-null    float64
 12  2025    234 non-null    float64
dtypes: float64(11), str(2)
memory usage: 27.1 KB


In [4]:
print(data_pib["Indicator Name"].unique())
data_pib.head(3)

<StringArray>
['GDP per capita (current US$)']
Length: 1, dtype: str


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,28440.041688,30082.158423,30654.485124,22664.370995,26827.344787,31000.571380,34897.618393,38590.565029,NaN,NaN
1,Africa Eastern and Southern,AFE,GDP per capita (current US$),NY.GDP.PCAP.CD,186.089515,186.909365,197.367876,225.400456,208.963066,226.836513,...,1529.923080,1553.617944,1508.031525,1351.503167,1560.894626,1675.902524,1571.132704,1628.227289,1722.38562,NaN
2,Afghanistan,AFG,GDP per capita (current US$),NY.GDP.PCAP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,525.469771,491.337221,496.602504,510.787063,356.496214,357.261153,413.757895,416.871146,NaN,NaN


In [5]:
anios = [str(a) for a in range(2015, 2026)]
data_pib = data_pib[["Country Name", "Country Code"] + anios]
print(data_pib.shape)

(266, 13)


- `range(2000, 2026)` → 2000…2025; `str(a)` → texto, porque las columnas se llaman `"2000"`.
- `[[ ... ]]` doble corchete = «quédate solo con estas columnas» (2 + 26).

**Debe salir:** `(266, 28)`.

## Paso 4 · Renombrar

In [6]:
data_pib = data_pib.rename(columns={"Country Name": "pais", "Country Code": "iso3"})

- `rename(columns={viejo: nuevo})` → mismos nombres que en el resto de datasets (la clave del merge es `iso3`).

## Paso 5 · Quitar agregados (Mundo, UE, grupos de renta…)

In [ ]:
equivalencias = pd.read_csv("../data/4_EQUIVALENCIAS_PAISES_ISO_3.csv")
data_pib = data_pib[data_pib["iso3"].isin(equivalencias["iso3"])]
print(data_pib.shape)

- Leemos tu tabla de países reales.
- `isin(...)` → `True` si el código está en la tabla; `df[máscara]` deja solo esas filas.

**Debe salir:** `(215, 28)`.

## Paso 6 · De ancho a largo con `melt`

In [ ]:
data_pib = data_pib.melt(
    id_vars=["pais", "iso3"],
    var_name="anio",
    value_name="pib_pc_usd"
)
print(data_pib.shape)

- `id_vars` → columnas fijas; `var_name` → nombre de la columna que recoge los años; `value_name` → la que recoge los valores.

**Debe salir:** `(5590, 4)` = 215 × 26.

## Paso 7 · `anio` a entero

In [ ]:
data_pib["anio"] = data_pib["anio"].astype(int)
print(data_pib.dtypes)

- Tras `melt` el año es texto; con texto el merge con turismo/vivienda no casaría.

**Debe salir:** `anio` `int64`, `pib_pc_usd` `float64`.

## Paso 8 · Ordenar (¡antes de calcular el crecimiento!)

In [ ]:
data_pib = data_pib.sort_values(["pais", "anio"]).reset_index(drop=True)
data_pib.head()

- `pct_change` compara cada fila con la **anterior**, así que dentro de cada país los años deben ir en orden. Por eso ordenamos antes.
- `reset_index(drop=True)` → renumera 0,1,2…

## Paso 9 · Calcular el crecimiento % anual por país

In [ ]:
data_pib["crecimiento_pib_pc_pct"] = data_pib.groupby("iso3")["pib_pc_usd"].pct_change(fill_method=None) * 100
data_pib[data_pib["iso3"] == "ESP"].head(4)

- `groupby("iso3")` → trabaja país a país. **Clave:** sin esto, la primera fila de España se compararía con la última de Albania.
- `["pib_pc_usd"].pct_change()` → `(valor_actual − valor_anterior) / valor_anterior`. Ejemplo: 100 → 110 da `0.10`.
- `fill_method=None` → si falta un dato, no lo rellena con el anterior (no inventamos).
- `* 100` → de proporción a porcentaje (0.10 → 10.0), igual que inflación y vivienda.
- La primera fila de cada país queda en `NaN`: no tiene año anterior.
- Es crecimiento **nominal en dólares corrientes**, y por eso se ve afectado por el tipo de cambio. Anótalo en la memoria como limitación.

## Paso 10 · Quitar NaN

In [ ]:
print(data_pib.isnull().sum())
data_pib = data_pib.dropna(subset=["crecimiento_pib_pc_pct"]).reset_index(drop=True)
print(data_pib.shape)

- `isnull().sum()` → huecos por columna. **Debe salir:** `pib_pc_usd` 203 y `crecimiento_pib_pc_pct` 418 (203 sin dato + 215 primeras filas de cada país).
- `dropna(subset=[...])` → quita filas sin crecimiento. Al quitarlas, `pib_pc_usd` también queda sin NaN.

**Debe salir:** `(5172, 5)`.

## Paso 11 · Comprobaciones

In [ ]:
print(data_pib["iso3"].nunique())
print(data_pib.groupby(["iso3", "anio"]).size().max())
print(data_pib["anio"].min(), data_pib["anio"].max())
data_pib[(data_pib["iso3"] == "ESP") & (data_pib["anio"] >= 2023)]

- **Debe salir:** `211` países, máximo `1` (sin duplicados), años `2001 2025` (2000 desaparece porque no tiene año anterior).
- España: 2023 → 10.47 · 2024 → 5.47 · 2025 → 9.34 (valores en %, dólares corrientes).

## Paso 12 · Guardar

In [7]:
data_pib.to_csv("../data/limpio_pib.csv", index=False)

- `index=False` → sin columna extra de índice. El merge leerá `limpio_pib.csv` directamente.

## Resumen de la lógica
`leer (skiprows) → columnas → renombrar → filtrar agregados → melt → astype → ordenar → groupby + pct_change → dropna → comprobar → guardar`

Patrón nuevo: **calcular variación por grupo = ordenar + groupby + pct_change**. Lo mismo servirá para el crecimiento del turismo.